# Avaliação walk-forward:XGBoost

Walk-forward é uma técnica de validação cruzada usada em séries temporais, onde o modelo é treinado em um conjunto de dados e testado em um conjunto subsequente. Isso é feito repetidamente, movendo a janela de treinamento e teste para frente no tempo.


In [1]:
!pip install -U catboost xgboost scikit-learn pandas pyarrow

^C


In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (average_precision_score, confusion_matrix,
                             precision_score, recall_score, f1_score)

In [ ]:
import os
pathKaggle = "../input/datasets/luisfilipelima/bancotransacao/"
path = "../../data/processed/"
pathDrive = "/content/drive/MyDrive/TreinosIAS/XGBoosting/dados/"
#path=pathDrive
#path = pathKaggle
path

'../../data/processed/'

In [ ]:
obt = pd.read_parquet(path+"obt_with_terminal_history.parquet")
obtTrain = pd.read_parquet(path+"obt_train.parquet")
obtTest = pd.read_parquet(path+"obt_validation.parquet")
obtTrain

,transaction_id,tx_datetime,tx_amount,card_id,terminal_id,terminal_category,hour,day_week,terminal_age_days,is_fraud,...,card_distinct_terminals_24h,seconds_since_last_card_tx,terminal_tx_count_1h,terminal_tx_count_24h,terminal_distinct_cards_24h,seconds_since_last_terminal_tx,card_history_count_30d,card_amount_mean_30d,card_amount_std_30d,amount_z_score_30d
0,63551,2018-01-03 14:46:39,13.790000,30469057923212,98,FreshMart,14,3,393.0,False,...,0,NaN,1,22,22,3210.0,0,NaN,NaN,NaN
1,160731,2018-01-07 16:31:49,11.020000,30469057923212,955,Elegant Carpets,16,0,104.0,False,...,0,351910.0,2,31,28,160.0,1,13.790000,NaN,NaN
2,264046,2018-01-12 05:53:34,37.709999,30469057923212,854,Vitality Gym,5,5,137.0,False,...,0,393705.0,2,24,24,1170.0,2,12.405000,1.958685,12.919379
3,264993,2018-01-12 06:50:19,25.290001,30469057923212,973,Gold Star Car Wash,6,5,103.0,False,...,1,3405.0,1,30,30,1386.0,3,20.840000,14.675349,0.303230
4,373789,2018-01-16 13:06:01,18.660000,30469057923212,786,Starlight Cinema,13,2,165.0,False,...,0,368142.0,1,23,23,1953.0,4,21.952500,12.187202,-0.270160
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3428899,3392898,2018-04-29 19:19:24,513.229980,4682776545183079364,1850,Elite Gaming,19,0,41.0,False,...,5,30208.0,0,22,22,4609.0,181,154.742320,164.837789,2.174791
3428900,3418568,2018-04-30 14:29:23,67.320000,4682776545183079364,931,FreshMart,14,1,226.0,False,...,1,68999.0,2,25,24,726.0,179,157.706536,167.560088,-0.539428
3428901,3419962,2018-04-30 15:08:18,56.209999,4682776545183079364,280,Creative Art Supplies,15,1,415.0,False,...,2,2335.0,2,22,22,1008.0,180,157.204389,167.227149,-0.603935
3428902,3420910,2018-04-30 15:37:27,244.830002,4682776545183079364,1850,Elite Gaming,15,1,42.0,False,...,3,1749.0,2,21,21,89.0,181,156.646408,166.930858,0.528264


In [ ]:
cat_features = ["terminal_category"]

features = [
    "tx_amount",
    "hour",
    "day_week",
    "terminal_age_days",
    "terminal_fraud_rate",
    "card_fraud_rate",
    "card_distinct_terminals",
    "terminal_new_for_card",
    "card_tx_count_1h",
    "card_tx_count_24h",
    "card_distinct_terminals_24h",
    "seconds_since_last_card_tx",
    "terminal_tx_count_1h",
    "terminal_tx_count_24h",
    "terminal_distinct_cards_24h",
    "seconds_since_last_terminal_tx",
    "card_history_count_30d",
    "card_amount_mean_30d",
    "card_amount_std_30d",
    "amount_z_score_30d",
    "terminal_category",
]
target = "is_fraud"
def prepararParaXGBoostging(df):
    df["datetime"] = pd.to_datetime(df["tx_datetime"], errors="raise")
    df[target] = df[target].astype(int)

    # XGB possui um suporte nativo para dados categoricos, tem que definir o tipo do dado como "category"
    df["terminal_category"] = (
        df["terminal_category"]
        .astype("object")
        .fillna("SEM_CATEGORIA")
        .astype("category")
)
    df = df.sort_values("tx_datetime").reset_index(drop=True)
    return df


    
obt = prepararParaXGBoostging(obt)
obtTrain = prepararParaXGBoostging(obtTrain)
obtTest = prepararParaXGBoostging(obtTest)


In [ ]:
from xgboost import XGBClassifier

In [ ]:
def calcular_lucro(TransacoesAceitas):
    if TransacoesAceitas.empty:
        print("Nenhuma transação aceita; lucro = 0.")
        return 0.0
    legitimas = TransacoesAceitas["is_fraud"].eq(0)
    fraudulentas = TransacoesAceitas["is_fraud"].eq(1)

    receita = (
        0.03 * TransacoesAceitas.loc[legitimas, "tx_amount"].sum()
    )

    prejuizo = (
        TransacoesAceitas.loc[fraudulentas, "tx_amount"].sum()
    )
    print("CALCULANDO MELHOR LUCRO...")
    print(f"VALOR:{float(receita - prejuizo)}")
    # Recusas têm resultado financeiro direto igual a zero.
    return float(receita - prejuizo).__round__(2)


def classificarTransacoes(x,y,prob,limiar):
    transacoes = x.copy()
    transacoes["is_fraud"] = np.asarray(y, dtype=int)
    transacoes["probabilidade_fraude"] = prob

    aceita = transacoes["probabilidade_fraude"] < limiar

    TransacoesAceitas = transacoes.loc[aceita].copy()
    TransacoesRecusadas = transacoes.loc[~aceita].copy() # pega todas menos as acceitas

    return TransacoesAceitas, TransacoesRecusadas

def encontrar_melhor_limiar(modelo, X_val, y_val):
    probabilidades = modelo.predict_proba(X_val)[:, 1]
    valores = X_val["tx_amount"].to_numpy()
    y = np.asarray(y_val, dtype=int)

    # Ordena da menor para a maior chance de fraude. Assim cada posição
    # representa o lucro acumulado ao aceitar as transações até aquele ponto.
    ordem = np.argsort(probabilidades, kind="stable")
    probabilidades_ordenadas = probabilidades[ordem]
    valores_ordenados = valores[ordem]
    y_ordenado = y[ordem]

    lucro_por_transacao = np.where(
        y_ordenado == 0,
        0.03 * valores_ordenados,
        -valores_ordenados,
    )
    lucro_acumulado = np.cumsum(lucro_por_transacao)

    
    fim_de_grupo = np.concatenate([
        probabilidades_ordenadas[1:] != probabilidades_ordenadas[:-1],
        [True],
    ])
    indices = np.flatnonzero(fim_de_grupo)
    lucros = lucro_acumulado[indices]

    
    lucros = np.concatenate([[0.0], lucros])
    melhor_posicao = int(np.argmax(lucros))
    melhor_lucro = float(lucros[melhor_posicao])

    if melhor_posicao == 0:
        melhor_limiar = np.nextafter(
            probabilidades_ordenadas[0], -np.inf
        )
    else:
        indice = indices[melhor_posicao - 1]
        melhor_limiar = np.nextafter(
            probabilidades_ordenadas[indice], np.inf
        )

    return float(melhor_limiar), melhor_lucro


In [ ]:
from sklearn.model_selection import TimeSeriesSplit
import optuna
tscv = TimeSeriesSplit(n_splits=10)


c:\Users\lf446\OneDrive\Documentos\GitHub\ti326m-trabalho3-equipe2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
SEED = 42
N_INIT_TRIALS = 10
N_TRIALS = 30
N_SPLITS = 10


Buscatrain = obtTrain.tail(300_000).copy()
X_Train = Buscatrain[features].copy().reset_index(drop=True)
y_Train = Buscatrain[target].copy().reset_index(drop=True)
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

fixed_params = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "tree_method": "hist",
    "device": "cuda",
    "enable_categorical": True,
    "random_state": SEED,
    "n_jobs": 4,
    "verbosity": 0,
}

def preparar_fold_xgb(X_fit, X_val):
    """Alinha categorias sem usar rótulos nem alterar a ordem temporal."""
    X_fit = X_fit.copy()
    X_val = X_val.copy()
    for coluna in cat_features:
        categorias = pd.Index(
            pd.concat([X_fit[coluna], X_val[coluna]], ignore_index=True).unique()
        )
        X_fit[coluna] = pd.Categorical(X_fit[coluna], categories=categorias)
        X_val[coluna] = pd.Categorical(X_val[coluna], categories=categorias)
    return X_fit, X_val

def objetivo(trial):
    params = {
        "n_estimators": 1000,
        "max_depth": trial.suggest_int("max_depth", 4, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.20, log=True),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 20.0, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 100.0, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 20.0, log=True),
    }

    lucros_folds = []
    prauc_folds = []
    limiares_folds = []

    for train_idx, val_idx in tscv.split(X_Train):
        X_fit, X_val = X_Train.iloc[train_idx], X_Train.iloc[val_idx]
        y_fit, y_val = y_Train.iloc[train_idx], y_Train.iloc[val_idx]
        X_fit, X_val = preparar_fold_xgb(X_fit, X_val)

        model = XGBClassifier(**fixed_params, **params)
        model.fit(X_fit, y_fit, verbose=False)

        probabilidades = model.predict_proba(X_val)[:, 1]
        # Cada limiar é escolhido somente dentro do fold de validação correspondente.
        limiar_fold, lucro_fold = encontrar_melhor_limiar(
            model, X_val, y_val
        )
        lucros_folds.append(lucro_fold)
        limiares_folds.append(limiar_fold)
        prauc_folds.append(average_precision_score(y_val, probabilidades))

    trial.set_user_attr("lucros_por_fold", lucros_folds)
    trial.set_user_attr("limiares_por_fold", limiares_folds)
    trial.set_user_attr("prauc_media", float(np.mean(prauc_folds)))
    return float(np.mean(lucros_folds))

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        seed=SEED,
        n_startup_trials=N_INIT_TRIALS,
    ),
)
study.optimize(
    objetivo,
    n_trials=N_TRIALS,
    n_jobs=1,
    show_progress_bar=True,
)




[I 2026-11-02 12:42:34,377] A new study created in memory with name: no-name-cd4cf7a6-23d1-4334-9cd6-b025573a17f6


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-11-02 12:43:05,042] Trial 0 finished with value: 49675.369921875 and parameters: {'max_depth': 5, 'learning_rate': 0.17254716573280354, 'min_child_weight': 8.960785365368121, 'gamma': 2.993292420985183, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'reg_alpha': 3.3323645788192616e-08, 'reg_lambda': 21.42302175774105, 'scale_pos_weight': 6.054365855469246}. Best is trial 0 with value: 49675.369921875.
[I 2026-11-02 12:43:53,726] Trial 1 finished with value: 50199.824609375 and parameters: {'max_depth': 7, 'learning_rate': 0.010636066512540286, 'min_child_weight': 18.276027831785726, 'gamma': 4.162213204002109, 'subsample': 0.6849356442713105, 'colsample_bytree': 0.6727299868828402, 'reg_alpha': 4.4734294104626844e-07, 'reg_lambda': 0.033205591037519584, 'scale_pos_weight': 4.816414530907083}. Best is trial 1 with value: 50199.824609375.
[I 2026-11-02 12:44:36,258] Trial 2 finished with value: 50107.12578125 and parameters: {'max_depth': 6, 'learning_rat

## Resultados

In [ ]:
print("Lucro médio dos folds da melhor configuração:", study.best_value)
print("Melhores hiperparâmetros:", study.best_params)
print("PRAUC médio nos folds:", study.best_trial.user_attrs["prauc_media"])
print("Lucros por fold:", study.best_trial.user_attrs["lucros_por_fold"])
print("Limiares por fold:", study.best_trial.user_attrs["limiares_por_fold"])

Lucro médio dos folds da melhor configuração: 50266.876953125
Melhores hiperparâmetros: {'max_depth': 4, 'learning_rate': 0.017840666160811192, 'min_child_weight': 12.236541108935361, 'gamma': 3.2179658727104634, 'subsample': 0.9416425154076182, 'colsample_bytree': 0.8238965108233988, 'reg_alpha': 1.8235272211937777e-07, 'reg_lambda': 0.984939511450461, 'scale_pos_weight': 1.1208330364341021}
PRAUC médio nos folds: 0.47274768610826773
Lucros por fold: [50751.671875, 51387.73828125, 49425.30078125, 48635.66015625, 49471.56640625, 48191.3671875, 51998.17578125, 49422.01953125, 52763.16015625, 50622.109375]
Limiares por fold: [0.13373799622058868, 0.09285085648298264, 0.08668658882379532, 0.04032743722200394, 0.1234259232878685, 0.140437513589859, 0.17430776357650757, 0.13782399892807007, 0.1429627239704132, 0.09859739989042282]


In [ ]:
import xgboost  as xgb
x_test, y_test = obtTest[features].copy(), obtTest[target].copy()
melhores_Parametros = study.best_params
print(study.best_params.user_attrs["n_trees"])
modelo_final = xgb.XGBClassifier(
    **fixed_params,
    **melhores_Parametros,
    n_estimators=study.best_trial.user_attrs["n_trees"],
)

modelo_final.fit(
    Buscatrain[features],
    Buscatrain[target],
    verbose=False,
)


limiar_final, lucro_limiar = encontrar_melhor_limiar(
    modelo_final,
    x_test,
    y_test,
)

print("Melhor limiar:", limiar_final)
print("Lucro:", lucro_limiar)

Melhor limiar: 0.08294454216957092
Lucro: 1738700.5


In [ ]:


probabilidades_teste = modelo_final.predict_proba(x_test)[:, 1]
predicoes_teste = (probabilidades_teste >= limiar_final).astype(int)

resultado_teste = x_test.copy()

resultado_teste["probabilidade_fraude"] = probabilidades_teste
resultado_teste["fraude_prevista"] = predicoes_teste
resultado_teste["fraude_real"] = y_test.to_numpy()

TransacoesAceitas, TransacoesRecusadas = classificarTransacoes(
    x_test,
    y_test,
    probabilidades_teste,
    limiar_final,
)

lucro_teste = calcular_lucro(TransacoesAceitas)
matriz = confusion_matrix(y_test, predicoes_teste)
precisao = precision_score(y_test, predicoes_teste, zero_division=0)
recall = recall_score(y_test, predicoes_teste, zero_division=0)
prauc_teste = average_precision_score(y_test, probabilidades_teste)

print("Matriz de confusão:")
print(matriz)
print("Lucro em maio:", lucro_teste)
print("Taxa de transações recusadas:", len(TransacoesRecusadas) / len(x_test))

res = resultado_teste.loc[
    (resultado_teste["fraude_real"] == 1)
    & (resultado_teste["fraude_prevista"] == 0)
]
print("erro / total")
print(f"{len(res)}/{len(resultado_teste.loc[resultado_teste["fraude_real"] == 1])}")

res = resultado_teste.loc[
    (resultado_teste["fraude_real"] == 1)
    & (resultado_teste["fraude_prevista"] == 1)
]
res
print("certo / total")
print(f"{len(res)}/{len(resultado_teste.loc[resultado_teste["fraude_real"] == 1])}")


CALCULANDO MELHOR LUCRO...
VALOR:1738742.75
Matriz de confusão:
[[1025315   16503]
 [   8173    5061]]
Lucro em maio: 1738742.75
Taxa de transações recusadas: 0.02043880301634422
erro / total
8173/13234
certo / total
5061/13234


In [ ]:
for parametro, valor in melhores_Parametros.items():
    print(f"{parametro}: {valor}")

max_depth: 4
learning_rate: 0.017840666160811192
min_child_weight: 12.236541108935361
gamma: 3.2179658727104634
subsample: 0.9416425154076182
colsample_bytree: 0.8238965108233988
reg_alpha: 1.8235272211937777e-07
reg_lambda: 0.984939511450461
scale_pos_weight: 1.1208330364341021
